In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
build_graphs.py  —  cyclone flood road-risk graph construction
================================================================
Builds one graph per city-event from per-cyclone road CSVs.

    Input   <City>_RoadFloodDataset_<Cyclone>.csv   (one per event)
    Output  graphs/<city>__<cyclone>.pt             (14 graphs)
            graphs/nodes_<city>.csv                 (road_id -> node idx)
            graphs/manifest.json                    (+ data card)

DESIGN DECISIONS AND WHY
------------------------

1. LINE GRAPH (segments = nodes, edges = shared intersections).
   The deliverable is a per-road risk score for a router, so roads
   must be nodes. A junction with k incident segments becomes a
   k-clique; real junctions are k=3-4 so this stays sparse, but
   malformed geometry can blow up quadratically, hence the cap.

2. HURDLE (TWO-HEAD) TARGET, not 4-class ordinal.
   Labels are ~99% dry, and "does water reach this road" and "how
   deep once it does" are different questions with different
   evidence. So: y_flood (binary, from SAR) + y_depth (continuous,
   from FwDET), each with its own mask.

   ⚠️ The two heads are NOT equally meaningful. FwDET derives depth
   from SAR extent + DEM arithmetic, and the DEM is also a model
   input — so the depth head is partly re-learning FwDET's own
   arithmetic rather than discovering physics. The FLOOD head is
   the genuine predictive contribution. Frame the paper that way.

3. SAR IS THE FLOOD LABEL; wse_assigned ONLY GATES DEPTH.
   FwDET's inward fill failing at a pixel says nothing about
   whether SAR saw water there — and the GEE scripts treat SAR
   extent as authoritative. Masking flood labels by wse_assigned
   (as an earlier version did) discards ~70% of positives from the
   head that needs them most.

4. PHYSICALLY-MOTIVATED AGGREGATION.
   Points -> segments uses min for HAND/elevation (the LOWEST point
   on a road is what floods first — a mean hides a dip), max for
   labels (a road is impassable if ANY part floods), mean for
   gradients and context.

5. CITY-TRANSFERABLE FEATURES ONLY.
   Under leave-one-city-out, any feature that identifies the city
   lets the model recognise where it is and then fail on the
   held-out one. Excluded for that reason: longitude, latitude,
   and ABSOLUTE elevation (Kolkata delta vs coastal Saurashtra have
   systematically different ranges). Elevation enters as a
   within-city percentile rank instead, which is scale-free and
   computable for an unseen city at inference. HAND is already
   relative (height above nearest drainage) and is kept as-is.

6. COLLINEAR / CONSTANT FEATURE PRUNING.
   effective_rain_mm and runoff_coeff are deterministic functions
   of rainfall_mm and curve_number (SCS-CN). If CN is a constant
   per event — it is, GEE writes ee.Image(CN_base) — then all four
   are perfectly collinear and three of them are redundant. The
   script measures within-graph variance and drops constants,
   keeping ONE event-severity scalar (rainfall_mm).
   Constants are not useless (they condition on storm severity),
   but with 14 events a set of them acts as an event ID the model
   can memorise. Keep one, drop the rest.

7. EDGE ATTRIBUTES: [delta_hand, delta_elev, midpoint_distance].
   A plain line graph says two roads touch but not how water would
   move between them. Elevation/HAND differences make message
   passing hydrologically meaningful at near-zero cost, and are
   usable by GINEConv / GATv2Conv(edge_dim=3). Models that ignore
   edge_attr still work unchanged.

8. NEIGHBOURHOOD TERRAIN FEATURES (hand_min_1hop, hand_mean_1hop,
   elev_rank_mean_1hop).
   These deliberately give the NON-GRAPH baselines access to
   hand-crafted neighbourhood context. That makes the headline
   comparison honest: not "graph vs no context" but "learned
   message passing vs hand-crafted aggregation" — the harder and
   more meaningful bar.

9. NO FEATURE SCALING HERE.
   Scaling is fold-dependent (fit on training cities only). Doing
   it at build time would leak test-city statistics into all folds.
   The harness owns it.

Install:  pip install pandas numpy torch torch_geometric
================================================================
"""

import os
import re
import glob
import json
import itertools
from collections import defaultdict

import numpy as np
import pandas as pd

try:
    import torch
    from torch_geometric.data import Data
    HAVE_PYG = True
except ImportError:
    HAVE_PYG = False


# ================================================================
# 0. CONFIG
# ================================================================

import os

INPUT_DIR = "/content/drive/MyDrive/csv_cyclone"
OUTPUT_DIR = "/content/drive/MyDrive/graphs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Graphs will be saved to:", OUTPUT_DIR)
print("Folder exists:", os.path.exists(OUTPUT_DIR))
# Display bands for reporting/serving only — NOT training targets.
# The model predicts P(flood) and depth; bands are applied at serve
# time so thresholds can change without retraining.
#
# ⚠️ REVIEW BEFORE ROUTING ANYONE. This is a vehicle-clearance
# judgement, not something the data decides. Values lean on commonly
# cited passenger-car figures (~0.15m before small cars struggle,
# ~0.30m where most stall) but are NOT from a specific standard.
# FwDET has no storm-surge term, so coastal depths are understated.
DEPTH_BANDS = [(0.15, "passable"), (0.30, "risky"), (np.inf, "impassable")]

# How each column collapses from sample points to a road segment
AGG_RULES = {
    # terrain — MIN: the lowest point on the segment floods first
    "hand_m":            "min",
    "elevation_m":       "min",
    # gradients / context — mean is representative
    "slope_deg":         "mean",
    "rainfall_mm":       "mean",
    "effective_rain_mm": "mean",
    "runoff_coeff":      "mean",
    "curve_number":      "mean",
    "road_length_m":     "mean",
    "longitude":         "mean",   # metadata only, not a feature
    "latitude":          "mean",   # metadata only, not a feature
    # labels — MAX: a road is impassable if ANY part of it floods
    "sar_flood_binary":  "max",
    "flood_depth_m":     "max",
    "wse_assigned":      "max",
}

# Candidate numeric features. Constants are pruned automatically
# (see §6 above); rainfall_mm is protected as the one retained
# event-severity scalar even if it is spatially constant.
CANDIDATE_FEATURES = [
    "hand_m",
    "slope_deg",
    "elev_rank",            # derived: within-city percentile
    "road_length_m",
    "degree",               # derived: node degree
    "hand_min_1hop",        # derived: neighbourhood terrain
    "hand_mean_1hop",
    "elev_rank_mean_1hop",
    "rainfall_mm",          # event-severity scalar (protected)
    "effective_rain_mm",    # pruned if constant (collinear w/ above)
    "runoff_coeff",         # pruned if constant
    "curve_number",         # pruned if constant
]
PROTECTED_FEATURES = {"rainfall_mm"}

# Computed after the graph exists (per city), so they are absent from
# the segment table when the constant check runs — keep them anyway.
DERIVED_FEATURES = {"degree", "hand_min_1hop", "hand_mean_1hop",
                    "elev_rank_mean_1hop", "elev_rank"}

# Never features: labels, masking inputs, city identifiers
EXCLUDED = {"sar_flood_binary", "flood_depth_m", "wse_assigned",
            "longitude", "latitude", "elevation_m"}

HIGHWAY_KEEP = ["motorway", "trunk", "primary", "secondary",
                "tertiary", "residential", "unclassified", "service"]

MAX_JUNCTION_DEGREE = 20      # skip implausible cliques
CONSTANT_STD_EPS    = 1e-9    # within-graph std below this = constant

FNAME_RE = re.compile(
    r"(?P<city>[A-Za-z]+)_RoadFloodDataset_(?P<cyclone>[A-Za-z]+)\.csv$")


# ================================================================
# 1. LOAD
# ================================================================

def canonicalize_road_id(road_id):
    """
    osmnx's 'drive' network is a directed MultiDiGraph: every two-way
    street becomes TWO directed edges, u_v_k and v_u_k, with mirrored
    geometry. Left as-is, ~95-97% of roads get sampled and stored
    twice under different road_ids — same terrain, same labels,
    silently double-counted, and every junction's clique roughly
    doubles (this is why mean degree came out ~8-9 instead of ~3-4).

    Orders endpoints numerically (not lexicographically — '10' < '9'
    as strings) so u_v_k and v_u_k collapse to the same canonical id
    before aggregation. Falls back to string order if endpoints
    aren't pure integers.
    """
    parts = str(road_id).split("_")
    if len(parts) < 3:
        return road_id
    u, v, k = parts[0], parts[1], "_".join(parts[2:])
    try:
        if int(u) > int(v):
            u, v = v, u
    except ValueError:
        if u > v:
            u, v = v, u
    return f"{u}_{v}_{k}"


def load_all_csvs(input_dir=INPUT_DIR):
    paths = sorted(glob.glob(os.path.join(input_dir, "*.csv")))
    if not paths:
        raise FileNotFoundError(f"No CSVs in {input_dir}")

    frames = []
    for p in paths:
        base = os.path.basename(p)
        if "ALL_CYCLONES" in base or base == "sampling_summary.csv":
            continue
        m = FNAME_RE.search(base)
        if not m:
            print(f"  skip (name pattern): {base}")
            continue
        df = pd.read_csv(p)
        df["city"] = m.group("city")
        if "cyclone" not in df.columns:
            df["cyclone"] = m.group("cyclone")
        frames.append(df)
        print(f"  {base}: {len(df):,} rows")

    all_df = pd.concat(frames, ignore_index=True)
    print(f"\n  {len(frames)} city-events, {len(all_df):,} sample points")

    # Collapse reverse-direction duplicates (see canonicalize_road_id).
    # Do this once here, before aggregation, so it applies uniformly —
    # aggregate_to_segments then merges both directions' sample points
    # into one segment automatically via the existing groupby.
    before = all_df["road_id"].nunique()
    all_df["road_id"] = all_df["road_id"].map(canonicalize_road_id)
    after = all_df["road_id"].nunique()
    if before > after:
        pct = 100 * (before - after) / before
        print(f"  canonicalized road_id: {before:,} -> {after:,} unique "
              f"({pct:.1f}% were reverse-direction duplicates, now merged)")

    return all_df


def aggregate_to_segments(df):
    agg = {c: rule for c, rule in AGG_RULES.items() if c in df.columns}
    missing = [c for c in AGG_RULES if c not in df.columns]
    if missing:
        print(f"  ⚠️  absent from CSVs: {missing}")

    for c in ("highway_type", "osm_name"):
        if c in df.columns:
            agg[c] = "first"

    seg = df.groupby(["city", "cyclone", "road_id"], as_index=False).agg(agg)
    print(f"  {len(df):,} points -> {len(seg):,} segment-events")
    return seg


# ================================================================
# 2. LINE GRAPH + EDGE ATTRIBUTES
# ================================================================

def parse_endpoints(road_id):
    """road_id is 'u_v_key' from osmnx; u and v are intersections."""
    parts = str(road_id).split("_")
    return (parts[0], parts[1]) if len(parts) >= 2 else (None, None)


def build_line_graph(segment_ids):
    idx_of = {rid: i for i, rid in enumerate(segment_ids)}

    incident = defaultdict(list)
    for rid in segment_ids:
        u, v = parse_endpoints(rid)
        if u is None:
            continue
        incident[u].append(idx_of[rid])
        if v != u:
            incident[v].append(idx_of[rid])

    src, dst, huge = [], [], 0
    for segs in incident.values():
        if len(segs) > MAX_JUNCTION_DEGREE:
            huge += 1
            continue
        for a, b in itertools.combinations(segs, 2):
            src += [a, b]
            dst += [b, a]

    if huge:
        print(f"  ⚠️  skipped {huge} junctions with >{MAX_JUNCTION_DEGREE} "
              f"incident segments (likely malformed geometry)")

    if not src:
        print("  ⚠️  NO EDGES — check road_id looks like 'u_v_key'")
        return (np.zeros((2, 0), np.int64), np.zeros(len(segment_ids), np.int64))

    # de-duplicate: two segments can share BOTH endpoints (parallel
    # ways, roundabout halves), which would otherwise double the edge
    edges = np.unique(np.vstack([src, dst]).T, axis=0).T.astype(np.int64)
    degree = np.bincount(edges[0], minlength=len(segment_ids)).astype(np.int64)

    n_iso = int((degree == 0).sum())
    md = degree.mean()
    print(f"  line graph: {len(segment_ids):,} nodes, {edges.shape[1]:,} "
          f"directed edges, mean degree {md:.2f}, {n_iso} isolated")
    if md > 8:
        print(f"  ⚠️  mean degree {md:.1f} is high for a road network "
              f"(expect ~3-4). Check road_id parsing for this city.")
    return edges, degree


def build_edge_attr(edge_index, hand, elev_rank, lon, lat):
    """
    [delta_hand, delta_elev_rank, midpoint_distance_km] per directed
    edge. Signed deltas (dst - src) so direction carries meaning:
    a negative delta_hand means the neighbour sits lower, i.e. water
    would run toward it.
    """
    s, d = edge_index[0], edge_index[1]
    d_hand = (hand[d] - hand[s]).astype(np.float32)
    d_elev = (elev_rank[d] - elev_rank[s]).astype(np.float32)

    # equirectangular approximation — fine at intra-city scale
    mlat = np.radians((lat[s] + lat[d]) / 2.0)
    dx = (lon[d] - lon[s]) * 111.32 * np.cos(mlat)
    dy = (lat[d] - lat[s]) * 110.57
    dist = np.sqrt(dx ** 2 + dy ** 2).astype(np.float32)

    return np.vstack([d_hand, d_elev, dist]).T  # [E, 3]


def neighbourhood_features(edge_index, n_nodes, hand, elev_rank):
    """
    1-hop aggregates, computed once per city (terrain is static).
    Isolated nodes fall back to their own value.
    Given to the tabular baselines too, so the GNN comparison is
    "learned message passing vs hand-crafted aggregation".
    """
    hand_min = hand.copy().astype(np.float64)
    hand_sum = np.zeros(n_nodes)
    elev_sum = np.zeros(n_nodes)
    cnt      = np.zeros(n_nodes)

    s, d = edge_index[0], edge_index[1]
    np.minimum.at(hand_min, s, hand[d])
    np.add.at(hand_sum, s, hand[d])
    np.add.at(elev_sum, s, elev_rank[d])
    np.add.at(cnt, s, 1.0)

    safe = np.maximum(cnt, 1.0)
    hand_mean = np.where(cnt > 0, hand_sum / safe, hand)
    elev_mean = np.where(cnt > 0, elev_sum / safe, elev_rank)
    return hand_min, hand_mean, elev_mean


# ================================================================
# 3. LABELS  (hurdle: flood head + depth head)
# ================================================================

def build_labels(ev):
    """
    FLOOD head — label is SAR, which the GEE pipeline treats as
    authoritative. wse_assigned is irrelevant here: FwDET failing to
    fill a pixel says nothing about whether SAR saw water.
    Masked only where SAR gave no observation at all.

    DEPTH head — only where SAR says flooded AND FwDET actually
    assigned a water surface. Elsewhere depth is not "0 m", it is
    unknown, and training on it would teach that flooded roads are
    dry.

    ⚠️ The depth head therefore trains on a biased subset: WSE is
    assigned near low-slope boundary anchors, so the model learns
    depth from the easier-to-fill regions. State this in limitations.
    """
    depth = ev["flood_depth_m"].fillna(0).to_numpy()
    sar   = ev["sar_flood_binary"].fillna(0).to_numpy()

    y_flood    = sar.astype(np.int64)
    mask_flood = ev["sar_flood_binary"].notna().to_numpy()

    if "wse_assigned" in ev.columns:
        assigned = ev["wse_assigned"].fillna(0).to_numpy() > 0
    else:
        assigned = depth > 0     # fallback: loses genuine shallow zeros
        print("    ⚠️  no wse_assigned column — depth mask falls back to "
              "depth>0, which discards true shallow readings")

    return y_flood, mask_flood, depth.astype(np.float32), (sar == 1) & assigned


# ================================================================
# 4. FEATURES
# ================================================================

def encode_highway(ev):
    """One-hot; column set forced identical across cities so an
    absent type in one city can't shift the feature dimension."""
    cols = [f"hw_{k}" for k in HIGHWAY_KEEP + ["other"]]
    if "highway_type" not in ev.columns:
        return pd.DataFrame(0.0, index=ev.index, columns=cols, dtype=np.float32)

    ht = (ev["highway_type"].astype(str)
          .str.split(";").str[0].str.strip().str.lower()
          .str.replace(r"^\[|\]$|'", "", regex=True))
    ht = ht.where(ht.isin(HIGHWAY_KEEP), "other")

    dummies = pd.get_dummies(ht, prefix="hw").astype(np.float32)
    for c in cols:
        if c not in dummies.columns:
            dummies[c] = 0.0
    return dummies[cols]


def report_constants(seg, candidates):
    """
    Measures within-graph spatial variance. A feature that is
    constant across every node of a graph carries only event-level
    information; with 14 events a set of them acts as an event ID.
    Returns the list to keep.

    NOTE: derived features (degree, *_1hop) do not exist on `seg`
    yet — they are computed per city once the graph is built — so
    they are kept unconditionally rather than being silently
    skipped by the `not in seg.columns` branch.
    """
    print("\nSpatial variance within graphs (std across nodes, "
          "averaged over events):")
    keep, dropped = [], []
    for c in candidates:
        if c not in seg.columns:
            if c in DERIVED_FEATURES:
                keep.append(c)
                print(f"  {c:22s} (derived later — kept)")
            else:
                print(f"  {c:22s} ABSENT from CSVs — skipped")
            continue
        s = seg.groupby(["city", "cyclone"])[c].std().mean()
        n = seg.groupby(["city", "cyclone"])[c].nunique().mean()
        const = (not np.isfinite(s)) or s < CONSTANT_STD_EPS
        tag = ""
        if const and c in PROTECTED_FEATURES:
            tag = "  CONSTANT — kept (event-severity scalar)"
            keep.append(c)
        elif const:
            tag = "  CONSTANT — dropped (redundant event ID)"
            dropped.append(c)
        else:
            keep.append(c)
        print(f"  {c:22s} std={s:12.6f}  uniq/graph={n:8.1f}{tag}")

    if dropped:
        print(f"\n  dropped {len(dropped)} constant feature(s): {dropped}")
        print("  (rainfall_mm retained as the single storm-severity signal;")
        print("   effective_rain_mm / runoff_coeff / curve_number are")
        print("   deterministic functions of it under SCS-CN, so keeping")
        print("   them adds collinearity, not information.)")
    return keep


# ================================================================
# 5. MAIN
# ================================================================

def main():
    print("Loading CSVs...")
    df = load_all_csvs()

    print("\nAggregating to segments...")
    seg = aggregate_to_segments(df)

    # Imputation: city median, never 0. A zero HAND or elevation is
    # exactly the artifact that produced absurd depths upstream.
    print("\nImputing NaNs (city median)...")
    for col in AGG_RULES:
        if col in seg.columns and seg[col].isna().any():
            n = int(seg[col].isna().sum())
            seg[col] = seg.groupby("city")[col].transform(
                lambda s: s.fillna(s.median()))
            seg[col] = seg[col].fillna(seg[col].median())
            print(f"  {col}: filled {n:,}")

    # Elevation -> within-city percentile rank. Scale-free, so it
    # transfers to an unseen city; absolute elevation would let the
    # model identify which city it is looking at.
    if "elevation_m" in seg.columns:
        seg["elev_rank"] = seg.groupby("city")["elevation_m"].rank(pct=True)

    features = report_constants(seg, CANDIDATE_FEATURES)

    feature_names_ref, manifest = None, []

    for city, city_df in seg.groupby("city"):
        print(f"\n{'━' * 62}\n{city}")

        # Topology is shared across a city's events — the road network
        # doesn't change between cyclones, only features and labels do.
        # Building once also guarantees identical node ordering.
        segment_ids = sorted(city_df["road_id"].unique())
        edge_index, degree = build_line_graph(segment_ids)

        # Static terrain, taken from the first event (identical across
        # events for a given city)
        first = (city_df.groupby("road_id").first().reindex(segment_ids))
        hand = first["hand_m"].to_numpy(np.float64)
        erank = (first["elev_rank"].to_numpy(np.float64)
                 if "elev_rank" in first.columns else np.zeros(len(segment_ids)))
        lon = first["longitude"].to_numpy(np.float64)
        lat = first["latitude"].to_numpy(np.float64)

        h_min, h_mean, e_mean = neighbourhood_features(
            edge_index, len(segment_ids), hand, erank)
        edge_attr = build_edge_attr(edge_index, hand, erank, lon, lat)

        # road_id -> node index, needed to paint predictions back onto
        # the map at serve time
        nodes_csv = os.path.join(OUTPUT_DIR, f"nodes_{city.lower()}.csv")
        pd.DataFrame({
            "node_idx": np.arange(len(segment_ids)),
            "road_id": segment_ids,
            "osm_name": first.get("osm_name", pd.Series(index=segment_ids)).values,
            "highway_type": first.get("highway_type", pd.Series(index=segment_ids)).values,
            "longitude": lon, "latitude": lat,
        }).to_csv(nodes_csv, index=False)

        for cyclone, ev in city_df.groupby("cyclone"):
            ev = ev.set_index("road_id").reindex(segment_ids)
            ev["degree"] = degree
            ev["hand_min_1hop"] = h_min
            ev["hand_mean_1hop"] = h_mean
            ev["elev_rank_mean_1hop"] = e_mean

            num = ev.reindex(columns=features).astype(np.float64).fillna(0.0)
            hw = encode_highway(ev)
            X = pd.concat([num.reset_index(drop=True),
                           hw.reset_index(drop=True)], axis=1)
            feature_names = list(X.columns)

            # Hard stop on feature drift — a mismatch here does not
            # error downstream, it silently trains on garbage.
            if feature_names_ref is None:
                feature_names_ref = feature_names
            elif feature_names != feature_names_ref:
                raise ValueError(
                    f"{city}/{cyclone} feature mismatch.\n"
                    f"  expected: {feature_names_ref}\n  got: {feature_names}")

            print(f"  {cyclone}:")
            y_flood, mask_flood, y_depth, mask_depth = build_labels(ev)

            fc = np.bincount(y_flood[mask_flood], minlength=2)
            pos_pct = 100 * fc[1] / max(fc.sum(), 1)
            print(f"    flood head: {mask_flood.sum():,}/{len(ev):,} usable | "
                  f"dry={fc[0]:,} flooded={fc[1]:,} ({pos_pct:.2f}%)")

            if mask_depth.sum():
                dv = y_depth[mask_depth]
                print(f"    depth head: {mask_depth.sum():,} usable | "
                      f"mean {dv.mean():.3f}m  p95 {np.percentile(dv,95):.3f}m  "
                      f"max {dv.max():.3f}m")
            else:
                print(f"    depth head: 0 usable")

            if fc[1] == 0:
                print("    ⚠️  NO positive flood labels — this event carries "
                      "no signal; decide whether to keep it before folds")
            if mask_depth.sum() < 30:
                print("    ⚠️  <30 usable depth labels — depth head will be "
                      "near-unsupervised for this event")

            stem = f"{city.lower()}__{cyclone.lower()}"
            if HAVE_PYG:
                data = Data(
                    x=torch.tensor(X.to_numpy(), dtype=torch.float),
                    edge_index=torch.tensor(edge_index, dtype=torch.long),
                    edge_attr=torch.tensor(edge_attr, dtype=torch.float),
                    y_flood=torch.tensor(y_flood, dtype=torch.long),
                    mask_flood=torch.tensor(mask_flood, dtype=torch.bool),
                    y_depth=torch.tensor(y_depth, dtype=torch.float),
                    mask_depth=torch.tensor(mask_depth, dtype=torch.bool),
                )
                data.city, data.cyclone = city, cyclone
                torch.save(data, os.path.join(OUTPUT_DIR, stem + ".pt"))
            else:
                np.savez_compressed(
                    os.path.join(OUTPUT_DIR, stem + ".npz"),
                    x=X.to_numpy().astype(np.float32),
                    edge_index=edge_index, edge_attr=edge_attr,
                    y_flood=y_flood, mask_flood=mask_flood,
                    y_depth=y_depth, mask_depth=mask_depth,
                    city=city, cyclone=cyclone)

            manifest.append({
                "city": city, "cyclone": cyclone,
                "n_nodes": len(segment_ids),
                "n_edges": int(edge_index.shape[1]),
                "mean_degree": round(float(degree.mean()), 2),
                "n_usable_flood": int(mask_flood.sum()),
                "n_flooded": int(fc[1]),
                "pct_flooded": round(float(pos_pct), 3),
                "n_usable_depth": int(mask_depth.sum()),
                "file": stem + (".pt" if HAVE_PYG else ".npz"),
            })

    # ── manifest + data card ──────────────────────────────────
    with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w") as f:
        json.dump({
            "task": "hurdle: y_flood (binary, SAR) + y_depth (metres, FwDET)",
            "feature_names": feature_names_ref,
            "edge_attr_names": ["delta_hand_m", "delta_elev_rank", "dist_km"],
            "display_bands": [b[1] for b in DEPTH_BANDS],
            "split": "leave-one-city-out",
            "notes": {
                "excluded_features": sorted(EXCLUDED),
                "why_excluded": "labels/masks, plus city identifiers "
                                "(lon/lat, absolute elevation) that break "
                                "leave-one-city-out generalisation",
                "scaling": "NOT applied — fold-dependent, fit on training "
                           "cities only in the harness",
                "depth_head_bias": "trains only where FwDET assigned a WSE, "
                                   "which favours low-slope boundary regions",
                "depth_head_caveat": "FwDET depth = SAR extent + DEM "
                                     "arithmetic, and the DEM is also an "
                                     "input; the depth head partly re-learns "
                                     "that arithmetic. The flood head is the "
                                     "genuine predictive contribution.",
                "no_storm_surge": "FwDET models terrestrial inundation only; "
                                  "coastal depths are understated for "
                                  "surge-driven events",
            },
            "graphs": manifest,
        }, f, indent=2)

    mf = pd.DataFrame(manifest)
    print(f"\n{'=' * 62}")
    print(mf.to_string(index=False))
    print(f"\nWrote {len(manifest)} graphs to {OUTPUT_DIR}/")
    print(f"Features ({len(feature_names_ref)}): {feature_names_ref}")
    if not HAVE_PYG:
        print("torch_geometric not installed — wrote .npz instead of .pt")

    dead = mf[mf.n_flooded == 0]
    thin = mf[(mf.n_flooded > 0) & (mf.pct_flooded < 0.5)]
    if len(dead):
        print(f"\n⚠️  {len(dead)} event(s) with zero flood labels:")
        print(dead[["city", "cyclone"]].to_string(index=False))
    if len(thin):
        print(f"\n⚠️  {len(thin)} event(s) below 0.5% flooded:")
        print(thin[["city", "cyclone", "pct_flooded"]].to_string(index=False))
    if len(dead) or len(thin):
        print("\nWatch Vayu (never made landfall — veered offshore) and Bulbul")
        print("(Sundarbans landfall, little rain in Kolkata proper). A null")
        print("event drags down its fold without adding information; dropping")
        print("one leaves Porbandar with 2 events.")

    print("\nNEXT — in the training harness:")
    print("  • leave-one-city-out; validation = one cyclone from the")
    print("    TRAINING cities, never the test city")
    print("  • fit the scaler on training cities only")
    print("  • pos_weight on the flood head; depth head trains only on")
    print("    mask_depth nodes")
    print("  • torch.load(..., weights_only=False) — PyG Data objects are")
    print("    not in torch>=2.6's default allowlist")
    print("=" * 62)


if __name__ == "__main__":
    main()

Graphs will be saved to: /content/drive/MyDrive/graphs
Folder exists: True
Loading CSVs...
  Chennai_RoadFloodDataset_Mandous.csv: 189,411 rows
  Chennai_RoadFloodDataset_Nivar.csv: 189,411 rows
  Chennai_RoadFloodDataset_Vardah.csv: 189,411 rows
  Kolkata_RoadFloodDataset_Amphan.csv: 304,248 rows
  Kolkata_RoadFloodDataset_Bulbul.csv: 304,248 rows
  Kolkata_RoadFloodDataset_Remal.csv: 304,248 rows
  Kolkata_RoadFloodDataset_Yaas.csv: 304,248 rows
  Porbandar_RoadFloodDataset_Biparjoy.csv: 23,910 rows
  Porbandar_RoadFloodDataset_Tauktae.csv: 23,910 rows
  Porbandar_RoadFloodDataset_Vayu.csv: 23,910 rows
  Puri_RoadFloodDataset_Dana.csv: 21,125 rows
  Puri_RoadFloodDataset_Fani.csv: 21,125 rows
  Puri_RoadFloodDataset_Titli.csv: 21,125 rows
  Puri_RoadFloodDataset_Yaas.csv: 21,125 rows

  14 city-events, 1,941,455 sample points
  canonicalized road_id: 483,370 -> 250,204 unique (48.2% were reverse-direction duplicates, now merged)

Aggregating to segments...
  1,941,455 points -> 900,5

In [ ]:
"""
build_graphs.py  —  cyclone flood road-risk graph construction
================================================================
Builds one graph per city-event from per-cyclone road CSVs.

    Input   <City>_RoadFloodDataset_<Cyclone>.csv   (one per event)
    Output  graphs/<city>__<cyclone>.pt             (14 graphs)
            graphs/nodes_<city>.csv                 (road_id -> node idx)
            graphs/manifest.json                    (+ data card)

DESIGN DECISIONS AND WHY
------------------------

1. LINE GRAPH (segments = nodes, edges = shared intersections).
   The deliverable is a per-road risk score for a router, so roads
   must be nodes. A junction with k incident segments becomes a
   k-clique; real junctions are k=3-4 so this stays sparse, but
   malformed geometry can blow up quadratically, hence the cap.

2. HURDLE (TWO-HEAD) TARGET, not 4-class ordinal.
   Labels are ~99% dry, and "does water reach this road" and "how
   deep once it does" are different questions with different
   evidence. So: y_flood (binary, from SAR) + y_depth (continuous,
   from FwDET), each with its own mask.

   ⚠️ The two heads are NOT equally meaningful. FwDET derives depth
   from SAR extent + DEM arithmetic, and the DEM is also a model
   input — so the depth head is partly re-learning FwDET's own
   arithmetic rather than discovering physics. The FLOOD head is
   the genuine predictive contribution. Frame the paper that way.

3. SAR IS THE FLOOD LABEL; wse_assigned ONLY GATES DEPTH.
   FwDET's inward fill failing at a pixel says nothing about
   whether SAR saw water there — and the GEE scripts treat SAR
   extent as authoritative. Masking flood labels by wse_assigned
   (as an earlier version did) discards ~70% of positives from the
   head that needs them most.

4. PHYSICALLY-MOTIVATED AGGREGATION.
   Points -> segments uses min for HAND/elevation (the LOWEST point
   on a road is what floods first — a mean hides a dip), max for
   labels (a road is impassable if ANY part floods), mean for
   gradients and context.

5. CITY-TRANSFERABLE FEATURES ONLY.
   Under leave-one-city-out, any feature that identifies the city
   lets the model recognise where it is and then fail on the
   held-out one. Excluded for that reason: longitude, latitude,
   and ABSOLUTE elevation (Kolkata delta vs coastal Saurashtra have
   systematically different ranges). Elevation enters as a
   within-city percentile rank instead, which is scale-free and
   computable for an unseen city at inference. HAND is already
   relative (height above nearest drainage) and is kept as-is.

6. COLLINEAR / CONSTANT FEATURE PRUNING.
   effective_rain_mm and runoff_coeff are deterministic functions
   of rainfall_mm and curve_number (SCS-CN). If CN is a constant
   per event — it is, GEE writes ee.Image(CN_base) — then all four
   are perfectly collinear and three of them are redundant. The
   script measures within-graph variance and drops constants,
   keeping ONE event-severity scalar (rainfall_mm).
   Constants are not useless (they condition on storm severity),
   but with 14 events a set of them acts as an event ID the model
   can memorise. Keep one, drop the rest.

7. EDGE ATTRIBUTES: [delta_hand, delta_elev, midpoint_distance].
   A plain line graph says two roads touch but not how water would
   move between them. Elevation/HAND differences make message
   passing hydrologically meaningful at near-zero cost, and are
   usable by GINEConv / GATv2Conv(edge_dim=3). Models that ignore
   edge_attr still work unchanged.

8. NEIGHBOURHOOD TERRAIN FEATURES (hand_min_1hop, hand_mean_1hop,
   elev_rank_mean_1hop).
   These deliberately give the NON-GRAPH baselines access to
   hand-crafted neighbourhood context. That makes the headline
   comparison honest: not "graph vs no context" but "learned
   message passing vs hand-crafted aggregation" — the harder and
   more meaningful bar.

9. NO FEATURE SCALING HERE.
   Scaling is fold-dependent (fit on training cities only). Doing
   it at build time would leak test-city statistics into all folds.
   The harness owns it.

Install:  pip install pandas numpy torch torch_geometric
================================================================
"""

import os
import re
import glob
import json
import itertools
from collections import defaultdict

import numpy as np
import pandas as pd

try:
    import torch
    from torch_geometric.data import Data
    HAVE_PYG = True
except ImportError:
    HAVE_PYG = False


# ================================================================
# 0. CONFIG
# ================================================================

INPUT_DIR  = "/content/drive/MyDrive/csv_cyclone"
OUTPUT_DIR = "./graphs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Display bands for reporting/serving only — NOT training targets.
# The model predicts P(flood) and depth; bands are applied at serve
# time so thresholds can change without retraining.
#
# ⚠️ REVIEW BEFORE ROUTING ANYONE. This is a vehicle-clearance
# judgement, not something the data decides. Values lean on commonly
# cited passenger-car figures (~0.15m before small cars struggle,
# ~0.30m where most stall) but are NOT from a specific standard.
# FwDET has no storm-surge term, so coastal depths are understated.
DEPTH_BANDS = [(0.15, "passable"), (0.30, "risky"), (np.inf, "impassable")]

# How each column collapses from sample points to a road segment
AGG_RULES = {
    # terrain — MIN: the lowest point on the segment floods first
    "hand_m":            "min",
    "elevation_m":       "min",
    # gradients / context — mean is representative
    "slope_deg":         "mean",
    "rainfall_mm":       "mean",
    "effective_rain_mm": "mean",
    "runoff_coeff":      "mean",
    "curve_number":      "mean",
    "road_length_m":     "mean",
    "longitude":         "mean",   # metadata only, not a feature
    "latitude":          "mean",   # metadata only, not a feature
    # labels — MAX: a road is impassable if ANY part of it floods
    "sar_flood_binary":  "max",
    "flood_depth_m":     "max",
    "wse_assigned":      "max",
}

# Candidate numeric features. Constants are pruned automatically
# (see §6 above); rainfall_mm is protected as the one retained
# event-severity scalar even if it is spatially constant.
CANDIDATE_FEATURES = [
    "hand_m",
    "slope_deg",
    "elev_rank",            # derived: within-city percentile
    "road_length_m",
    "degree",               # derived: node degree
    "hand_min_1hop",        # derived: neighbourhood terrain
    "hand_mean_1hop",
    "elev_rank_mean_1hop",
    "rainfall_mm",          # event-severity scalar (protected)
    "effective_rain_mm",    # pruned if constant (collinear w/ above)
    "runoff_coeff",         # pruned if constant
    "curve_number",         # pruned if constant
]
PROTECTED_FEATURES = {"rainfall_mm"}

# Computed after the graph exists (per city), so they are absent from
# the segment table when the constant check runs — keep them anyway.
DERIVED_FEATURES = {"degree", "hand_min_1hop", "hand_mean_1hop",
                    "elev_rank_mean_1hop", "elev_rank"}

# Never features: labels, masking inputs, city identifiers
EXCLUDED = {"sar_flood_binary", "flood_depth_m", "wse_assigned",
            "longitude", "latitude", "elevation_m"}

HIGHWAY_KEEP = ["motorway", "trunk", "primary", "secondary",
                "tertiary", "residential", "unclassified", "service"]

MAX_JUNCTION_DEGREE = 20      # skip implausible cliques
CONSTANT_STD_EPS    = 1e-9    # within-graph std below this = constant

FNAME_RE = re.compile(
    r"(?P<city>[A-Za-z]+)_RoadFloodDataset_(?P<cyclone>[A-Za-z]+)\.csv$")


# ================================================================
# 1. LOAD
# ================================================================

def canonicalize_road_id(road_id):
    """
    osmnx's 'drive' network is a directed MultiDiGraph: every two-way
    street becomes TWO directed edges, u_v_k and v_u_k, with mirrored
    geometry. Left as-is, ~95-97% of roads get sampled and stored
    twice under different road_ids — same terrain, same labels,
    silently double-counted, and every junction's clique roughly
    doubles (this is why mean degree came out ~8-9 instead of ~3-4).

    Orders endpoints numerically (not lexicographically — '10' < '9'
    as strings) so u_v_k and v_u_k collapse to the same canonical id
    before aggregation. Falls back to string order if endpoints
    aren't pure integers.
    """
    parts = str(road_id).split("_")
    if len(parts) < 3:
        return road_id
    u, v, k = parts[0], parts[1], "_".join(parts[2:])
    try:
        if int(u) > int(v):
            u, v = v, u
    except ValueError:
        if u > v:
            u, v = v, u
    return f"{u}_{v}_{k}"


def load_all_csvs(input_dir=INPUT_DIR):
    paths = sorted(glob.glob(os.path.join(input_dir, "*.csv")))
    if not paths:
        raise FileNotFoundError(f"No CSVs in {input_dir}")

    frames = []
    for p in paths:
        base = os.path.basename(p)
        if "ALL_CYCLONES" in base or base == "sampling_summary.csv":
            continue
        m = FNAME_RE.search(base)
        if not m:
            print(f"  skip (name pattern): {base}")
            continue
        df = pd.read_csv(p)
        df["city"] = m.group("city")
        if "cyclone" not in df.columns:
            df["cyclone"] = m.group("cyclone")
        frames.append(df)
        print(f"  {base}: {len(df):,} rows")

    all_df = pd.concat(frames, ignore_index=True)
    print(f"\n  {len(frames)} city-events, {len(all_df):,} sample points")

    # Collapse reverse-direction duplicates (see canonicalize_road_id).
    # Do this once here, before aggregation, so it applies uniformly —
    # aggregate_to_segments then merges both directions' sample points
    # into one segment automatically via the existing groupby.
    before = all_df["road_id"].nunique()
    all_df["road_id"] = all_df["road_id"].map(canonicalize_road_id)
    after = all_df["road_id"].nunique()
    if before > after:
        pct = 100 * (before - after) / before
        print(f"  canonicalized road_id: {before:,} -> {after:,} unique "
              f"({pct:.1f}% were reverse-direction duplicates, now merged)")

    return all_df


def aggregate_to_segments(df):
    agg = {c: rule for c, rule in AGG_RULES.items() if c in df.columns}
    missing = [c for c in AGG_RULES if c not in df.columns]
    if missing:
        print(f"  ⚠️  absent from CSVs: {missing}")

    for c in ("highway_type", "osm_name"):
        if c in df.columns:
            agg[c] = "first"

    seg = df.groupby(["city", "cyclone", "road_id"], as_index=False).agg(agg)
    print(f"  {len(df):,} points -> {len(seg):,} segment-events")
    return seg


# ================================================================
# 2. LINE GRAPH + EDGE ATTRIBUTES
# ================================================================

def parse_endpoints(road_id):
    """road_id is 'u_v_key' from osmnx; u and v are intersections."""
    parts = str(road_id).split("_")
    return (parts[0], parts[1]) if len(parts) >= 2 else (None, None)


def build_line_graph(segment_ids):
    idx_of = {rid: i for i, rid in enumerate(segment_ids)}

    incident = defaultdict(list)
    for rid in segment_ids:
        u, v = parse_endpoints(rid)
        if u is None:
            continue
        incident[u].append(idx_of[rid])
        if v != u:
            incident[v].append(idx_of[rid])

    src, dst, huge = [], [], 0
    for segs in incident.values():
        if len(segs) > MAX_JUNCTION_DEGREE:
            huge += 1
            continue
        for a, b in itertools.combinations(segs, 2):
            src += [a, b]
            dst += [b, a]

    if huge:
        print(f"  ⚠️  skipped {huge} junctions with >{MAX_JUNCTION_DEGREE} "
              f"incident segments (likely malformed geometry)")

    if not src:
        print("  ⚠️  NO EDGES — check road_id looks like 'u_v_key'")
        return (np.zeros((2, 0), np.int64), np.zeros(len(segment_ids), np.int64))

    # de-duplicate: two segments can share BOTH endpoints (parallel
    # ways, roundabout halves), which would otherwise double the edge
    edges = np.unique(np.vstack([src, dst]).T, axis=0).T.astype(np.int64)
    degree = np.bincount(edges[0], minlength=len(segment_ids)).astype(np.int64)

    n_iso = int((degree == 0).sum())
    md = degree.mean()
    print(f"  line graph: {len(segment_ids):,} nodes, {edges.shape[1]:,} "
          f"directed edges, mean degree {md:.2f}, {n_iso} isolated")
    if md > 8:
        print(f"  ⚠️  mean degree {md:.1f} is high for a road network "
              f"(expect ~3-4). Check road_id parsing for this city.")
    return edges, degree


def build_edge_attr(edge_index, hand, elev_rank, lon, lat):
    """
    [delta_hand, delta_elev_rank, midpoint_distance_km] per directed
    edge. Signed deltas (dst - src) so direction carries meaning:
    a negative delta_hand means the neighbour sits lower, i.e. water
    would run toward it.
    """
    s, d = edge_index[0], edge_index[1]
    d_hand = (hand[d] - hand[s]).astype(np.float32)
    d_elev = (elev_rank[d] - elev_rank[s]).astype(np.float32)

    # equirectangular approximation — fine at intra-city scale
    mlat = np.radians((lat[s] + lat[d]) / 2.0)
    dx = (lon[d] - lon[s]) * 111.32 * np.cos(mlat)
    dy = (lat[d] - lat[s]) * 110.57
    dist = np.sqrt(dx ** 2 + dy ** 2).astype(np.float32)

    return np.vstack([d_hand, d_elev, dist]).T  # [E, 3]


def neighbourhood_features(edge_index, n_nodes, hand, elev_rank):
    """
    1-hop aggregates, computed once per city (terrain is static).
    Isolated nodes fall back to their own value.
    Given to the tabular baselines too, so the GNN comparison is
    "learned message passing vs hand-crafted aggregation".
    """
    hand_min = hand.copy().astype(np.float64)
    hand_sum = np.zeros(n_nodes)
    elev_sum = np.zeros(n_nodes)
    cnt      = np.zeros(n_nodes)

    s, d = edge_index[0], edge_index[1]
    np.minimum.at(hand_min, s, hand[d])
    np.add.at(hand_sum, s, hand[d])
    np.add.at(elev_sum, s, elev_rank[d])
    np.add.at(cnt, s, 1.0)

    safe = np.maximum(cnt, 1.0)
    hand_mean = np.where(cnt > 0, hand_sum / safe, hand)
    elev_mean = np.where(cnt > 0, elev_sum / safe, elev_rank)
    return hand_min, hand_mean, elev_mean


# ================================================================
# 3. LABELS  (hurdle: flood head + depth head)
# ================================================================

def build_labels(ev):
    """
    FLOOD head — label is SAR, which the GEE pipeline treats as
    authoritative. wse_assigned is irrelevant here: FwDET failing to
    fill a pixel says nothing about whether SAR saw water.
    Masked only where SAR gave no observation at all.

    DEPTH head — only where SAR says flooded AND FwDET actually
    assigned a water surface. Elsewhere depth is not "0 m", it is
    unknown, and training on it would teach that flooded roads are
    dry.

    ⚠️ The depth head therefore trains on a biased subset: WSE is
    assigned near low-slope boundary anchors, so the model learns
    depth from the easier-to-fill regions. State this in limitations.
    """
    depth = ev["flood_depth_m"].fillna(0).to_numpy()
    sar   = ev["sar_flood_binary"].fillna(0).to_numpy()

    y_flood    = sar.astype(np.int64)
    mask_flood = ev["sar_flood_binary"].notna().to_numpy()

    if "wse_assigned" in ev.columns:
        assigned = ev["wse_assigned"].fillna(0).to_numpy() > 0
    else:
        assigned = depth > 0     # fallback: loses genuine shallow zeros
        print("    ⚠️  no wse_assigned column — depth mask falls back to "
              "depth>0, which discards true shallow readings")

    return y_flood, mask_flood, depth.astype(np.float32), (sar == 1) & assigned


# ================================================================
# 4. FEATURES
# ================================================================

def encode_highway(ev):
    """One-hot; column set forced identical across cities so an
    absent type in one city can't shift the feature dimension."""
    cols = [f"hw_{k}" for k in HIGHWAY_KEEP + ["other"]]
    if "highway_type" not in ev.columns:
        return pd.DataFrame(0.0, index=ev.index, columns=cols, dtype=np.float32)

    ht = (ev["highway_type"].astype(str)
          .str.split(";").str[0].str.strip().str.lower()
          .str.replace(r"^\[|\]$|'", "", regex=True))
    ht = ht.where(ht.isin(HIGHWAY_KEEP), "other")

    dummies = pd.get_dummies(ht, prefix="hw").astype(np.float32)
    for c in cols:
        if c not in dummies.columns:
            dummies[c] = 0.0
    return dummies[cols]


def report_constants(seg, candidates):
    """
    Measures within-graph spatial variance. A feature that is
    constant across every node of a graph carries only event-level
    information; with 14 events a set of them acts as an event ID.
    Returns the list to keep.

    NOTE: derived features (degree, *_1hop) do not exist on `seg`
    yet — they are computed per city once the graph is built — so
    they are kept unconditionally rather than being silently
    skipped by the `not in seg.columns` branch.
    """
    print("\nSpatial variance within graphs (std across nodes, "
          "averaged over events):")
    keep, dropped = [], []
    for c in candidates:
        if c not in seg.columns:
            if c in DERIVED_FEATURES:
                keep.append(c)
                print(f"  {c:22s} (derived later — kept)")
            else:
                print(f"  {c:22s} ABSENT from CSVs — skipped")
            continue
        s = seg.groupby(["city", "cyclone"])[c].std().mean()
        n = seg.groupby(["city", "cyclone"])[c].nunique().mean()
        const = (not np.isfinite(s)) or s < CONSTANT_STD_EPS
        tag = ""
        if const and c in PROTECTED_FEATURES:
            tag = "  CONSTANT — kept (event-severity scalar)"
            keep.append(c)
        elif const:
            tag = "  CONSTANT — dropped (redundant event ID)"
            dropped.append(c)
        else:
            keep.append(c)
        print(f"  {c:22s} std={s:12.6f}  uniq/graph={n:8.1f}{tag}")

    if dropped:
        print(f"\n  dropped {len(dropped)} constant feature(s): {dropped}")
        print("  (rainfall_mm retained as the single storm-severity signal;")
        print("   effective_rain_mm / runoff_coeff / curve_number are")
        print("   deterministic functions of it under SCS-CN, so keeping")
        print("   them adds collinearity, not information.)")
    return keep


# ================================================================
# 5. MAIN
# ================================================================

def main():
    print("Loading CSVs...")
    df = load_all_csvs()

    print("\nAggregating to segments...")
    seg = aggregate_to_segments(df)

    # Imputation: city median, never 0. A zero HAND or elevation is
    # exactly the artifact that produced absurd depths upstream.
    print("\nImputing NaNs (city median)...")
    for col in AGG_RULES:
        if col in seg.columns and seg[col].isna().any():
            n = int(seg[col].isna().sum())
            seg[col] = seg.groupby("city")[col].transform(
                lambda s: s.fillna(s.median()))
            seg[col] = seg[col].fillna(seg[col].median())
            print(f"  {col}: filled {n:,}")

    # Elevation -> within-city percentile rank. Scale-free, so it
    # transfers to an unseen city; absolute elevation would let the
    # model identify which city it is looking at.
    if "elevation_m" in seg.columns:
        seg["elev_rank"] = seg.groupby("city")["elevation_m"].rank(pct=True)

    features = report_constants(seg, CANDIDATE_FEATURES)

    feature_names_ref, manifest = None, []

    for city, city_df in seg.groupby("city"):
        print(f"\n{'━' * 62}\n{city}")

        # Topology is shared across a city's events — the road network
        # doesn't change between cyclones, only features and labels do.
        # Building once also guarantees identical node ordering.
        segment_ids = sorted(city_df["road_id"].unique())
        edge_index, degree = build_line_graph(segment_ids)

        # Static terrain, taken from the first event (identical across
        # events for a given city)
        first = (city_df.groupby("road_id").first().reindex(segment_ids))
        hand = first["hand_m"].to_numpy(np.float64)
        erank = (first["elev_rank"].to_numpy(np.float64)
                 if "elev_rank" in first.columns else np.zeros(len(segment_ids)))
        lon = first["longitude"].to_numpy(np.float64)
        lat = first["latitude"].to_numpy(np.float64)

        h_min, h_mean, e_mean = neighbourhood_features(
            edge_index, len(segment_ids), hand, erank)
        edge_attr = build_edge_attr(edge_index, hand, erank, lon, lat)

        # road_id -> node index, needed to paint predictions back onto
        # the map at serve time
        nodes_csv = os.path.join(OUTPUT_DIR, f"nodes_{city.lower()}.csv")
        pd.DataFrame({
            "node_idx": np.arange(len(segment_ids)),
            "road_id": segment_ids,
            "osm_name": first.get("osm_name", pd.Series(index=segment_ids)).values,
            "highway_type": first.get("highway_type", pd.Series(index=segment_ids)).values,
            "longitude": lon, "latitude": lat,
        }).to_csv(nodes_csv, index=False)

        for cyclone, ev in city_df.groupby("cyclone"):
            ev = ev.set_index("road_id").reindex(segment_ids)
            ev["degree"] = degree
            ev["hand_min_1hop"] = h_min
            ev["hand_mean_1hop"] = h_mean
            ev["elev_rank_mean_1hop"] = e_mean

            num = ev.reindex(columns=features).astype(np.float64).fillna(0.0)
            hw = encode_highway(ev)
            X = pd.concat([num.reset_index(drop=True),
                           hw.reset_index(drop=True)], axis=1)
            feature_names = list(X.columns)

            # Hard stop on feature drift — a mismatch here does not
            # error downstream, it silently trains on garbage.
            if feature_names_ref is None:
                feature_names_ref = feature_names
            elif feature_names != feature_names_ref:
                raise ValueError(
                    f"{city}/{cyclone} feature mismatch.\n"
                    f"  expected: {feature_names_ref}\n  got: {feature_names}")

            print(f"  {cyclone}:")
            y_flood, mask_flood, y_depth, mask_depth = build_labels(ev)

            fc = np.bincount(y_flood[mask_flood], minlength=2)
            pos_pct = 100 * fc[1] / max(fc.sum(), 1)
            print(f"    flood head: {mask_flood.sum():,}/{len(ev):,} usable | "
                  f"dry={fc[0]:,} flooded={fc[1]:,} ({pos_pct:.2f}%)")

            if mask_depth.sum():
                dv = y_depth[mask_depth]
                print(f"    depth head: {mask_depth.sum():,} usable | "
                      f"mean {dv.mean():.3f}m  p95 {np.percentile(dv,95):.3f}m  "
                      f"max {dv.max():.3f}m")
            else:
                print(f"    depth head: 0 usable")

            if fc[1] == 0:
                print("    ⚠️  NO positive flood labels — this event carries "
                      "no signal; decide whether to keep it before folds")
            if mask_depth.sum() < 30:
                print("    ⚠️  <30 usable depth labels — depth head will be "
                      "near-unsupervised for this event")

            stem = f"{city.lower()}__{cyclone.lower()}"
            if HAVE_PYG:
                data = Data(
                    x=torch.tensor(X.to_numpy(), dtype=torch.float),
                    edge_index=torch.tensor(edge_index, dtype=torch.long),
                    edge_attr=torch.tensor(edge_attr, dtype=torch.float),
                    y_flood=torch.tensor(y_flood, dtype=torch.long),
                    mask_flood=torch.tensor(mask_flood, dtype=torch.bool),
                    y_depth=torch.tensor(y_depth, dtype=torch.float),
                    mask_depth=torch.tensor(mask_depth, dtype=torch.bool),
                )
                data.city, data.cyclone = city, cyclone
                torch.save(data, os.path.join(OUTPUT_DIR, stem + ".pt"))
            else:
                np.savez_compressed(
                    os.path.join(OUTPUT_DIR, stem + ".npz"),
                    x=X.to_numpy().astype(np.float32),
                    edge_index=edge_index, edge_attr=edge_attr,
                    y_flood=y_flood, mask_flood=mask_flood,
                    y_depth=y_depth, mask_depth=mask_depth,
                    city=city, cyclone=cyclone)

            manifest.append({
                "city": city, "cyclone": cyclone,
                "n_nodes": len(segment_ids),
                "n_edges": int(edge_index.shape[1]),
                "mean_degree": round(float(degree.mean()), 2),
                "n_usable_flood": int(mask_flood.sum()),
                "n_flooded": int(fc[1]),
                "pct_flooded": round(float(pos_pct), 3),
                "n_usable_depth": int(mask_depth.sum()),
                "file": stem + (".pt" if HAVE_PYG else ".npz"),
            })

    # ── manifest + data card ──────────────────────────────────
    with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w") as f:
        json.dump({
            "task": "hurdle: y_flood (binary, SAR) + y_depth (metres, FwDET)",
            "feature_names": feature_names_ref,
            "edge_attr_names": ["delta_hand_m", "delta_elev_rank", "dist_km"],
            "display_bands": [b[1] for b in DEPTH_BANDS],
            "split": "leave-one-city-out",
            "notes": {
                "excluded_features": sorted(EXCLUDED),
                "why_excluded": "labels/masks, plus city identifiers "
                                "(lon/lat, absolute elevation) that break "
                                "leave-one-city-out generalisation",
                "scaling": "NOT applied — fold-dependent, fit on training "
                           "cities only in the harness",
                "depth_head_bias": "trains only where FwDET assigned a WSE, "
                                   "which favours low-slope boundary regions",
                "depth_head_caveat": "FwDET depth = SAR extent + DEM "
                                     "arithmetic, and the DEM is also an "
                                     "input; the depth head partly re-learns "
                                     "that arithmetic. The flood head is the "
                                     "genuine predictive contribution.",
                "no_storm_surge": "FwDET models terrestrial inundation only; "
                                  "coastal depths are understated for "
                                  "surge-driven events",
            },
            "graphs": manifest,
        }, f, indent=2)

    mf = pd.DataFrame(manifest)
    print(f"\n{'=' * 62}")
    print(mf.to_string(index=False))
    print(f"\nWrote {len(manifest)} graphs to {OUTPUT_DIR}/")
    print(f"Features ({len(feature_names_ref)}): {feature_names_ref}")
    if not HAVE_PYG:
        print("torch_geometric not installed — wrote .npz instead of .pt")

    dead = mf[mf.n_flooded == 0]
    thin = mf[(mf.n_flooded > 0) & (mf.pct_flooded < 0.5)]
    if len(dead):
        print(f"\n⚠️  {len(dead)} event(s) with zero flood labels:")
        print(dead[["city", "cyclone"]].to_string(index=False))
    if len(thin):
        print(f"\n⚠️  {len(thin)} event(s) below 0.5% flooded:")
        print(thin[["city", "cyclone", "pct_flooded"]].to_string(index=False))
    if len(dead) or len(thin):
        print("\nWatch Vayu (never made landfall — veered offshore) and Bulbul")
        print("(Sundarbans landfall, little rain in Kolkata proper). A null")
        print("event drags down its fold without adding information; dropping")
        print("one leaves Porbandar with 2 events.")

    print("\nNEXT — in the training harness:")
    print("  • leave-one-city-out; validation = one cyclone from the")
    print("    TRAINING cities, never the test city")
    print("  • fit the scaler on training cities only")
    print("  • pos_weight on the flood head; depth head trains only on")
    print("    mask_depth nodes")
    print("  • torch.load(..., weights_only=False) — PyG Data objects are")
    print("    not in torch>=2.6's default allowlist")
    print("=" * 62)


if __name__ == "__main__":
    main()
import shutil
import os

DRIVE_OUTPUT = "/content/drive/MyDrive/Cyclone_Dataset/graphs"

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for file in os.listdir("./graphs"):
    shutil.copy2(
        os.path.join("./graphs", file),
        os.path.join(DRIVE_OUTPUT, file)
    )

print("Copied to:", DRIVE_OUTPUT)
print(os.listdir(DRIVE_OUTPUT))

Loading CSVs...
  Chennai_RoadFloodDataset_Mandous.csv: 189,411 rows
  Chennai_RoadFloodDataset_Nivar.csv: 189,411 rows
  Chennai_RoadFloodDataset_Vardah.csv: 189,411 rows
  Kolkata_RoadFloodDataset_Amphan.csv: 304,248 rows
  Kolkata_RoadFloodDataset_Bulbul.csv: 304,248 rows
  Kolkata_RoadFloodDataset_Remal.csv: 304,248 rows
  Kolkata_RoadFloodDataset_Yaas.csv: 304,248 rows
  Porbandar_RoadFloodDataset_Biparjoy.csv: 23,910 rows
  Porbandar_RoadFloodDataset_Tauktae.csv: 23,910 rows
  Porbandar_RoadFloodDataset_Vayu.csv: 23,910 rows
  Puri_RoadFloodDataset_Dana.csv: 21,125 rows
  Puri_RoadFloodDataset_Fani.csv: 21,125 rows
  Puri_RoadFloodDataset_Titli.csv: 21,125 rows
  Puri_RoadFloodDataset_Yaas.csv: 21,125 rows

  14 city-events, 1,941,455 sample points
  canonicalized road_id: 483,370 -> 250,204 unique (48.2% were reverse-direction duplicates, now merged)

Aggregating to segments...
  1,941,455 points -> 900,538 segment-events

Imputing NaNs (city median)...

Spatial variance within 

In [ ]:
"""
train_gat.py  —  GATv2 for cyclone flood road-risk   [v2, patched]
================================================================
Trains the two-head (hurdle) model produced by build_graphs_v2.py:
    flood head  ->  P(this road floods at all)     [binary, from SAR]
    depth head  ->  how deep, given it floods      [metres, from FwDET]

Evaluation is LEAVE-ONE-CITY-OUT: train on three cities, test on
the fourth. That is the question that matters — will this work in a
city the model has never seen?

WHY NOT A RANDOM SPLIT: adjacent road segments are near-identical
(same terrain, same storm, they touch). A random node split puts a
node's own neighbours in the training set, the model effectively
sees the answer, and the score becomes meaningless. Never do it.

----------------------------------------------------------------
CHANGES IN v2 (all of these were real bugs or real methodology
problems in v1 — each is marked [FIX n] at the site)

 1. fit_scaler used .clamp(min=1e-6). Any feature with ~zero
    variance across the TRAINING cities but nonzero in the held-out
    city became (x - 0) / 1e-6 ~= 1e6 and blew the model up. The
    one-hot hw_* columns and curve_number (1.3 unique values per
    graph) are the exposed ones. Measured on a planted case:
    PR-AUC 0.234 -> 0.532, depth MAE 19.46 m -> 0.235 m.
    Now: floor the std at 1e-2 (center-only for near-constant
    columns) and clip scaled features to +/-10. Floored columns
    are printed so you can see it happen.

 2. Metrics were reported at threshold 0.5 while training with
    pos_weight ~= 10-11. pos_weight deliberately shifts the logits,
    so 0.5 is not the operating point — measured recall 0.74 at
    precision 0.28. The threshold is now chosen on VALIDATION and
    test metrics are reported at both 0.5 and the tuned threshold.

 3. Model selection. v1 used "last cyclone alphabetically" which
    resolved to Yaas in all four folds, and for test=Kolkata left
    val = Puri/Yaas alone (the 31.9%-flooded outlier). Worse, val
    graphs share IDENTICAL node sets with training graphs, so val
    PR-AUC is partly memorisation (0.73 val vs 0.36 test on the
    Kolkata fold). Default is now --val-mode fixed: a fixed step
    budget, no early stopping, final model reported. Val is still
    computed and logged, just not used to pick the checkpoint.
    --val-mode pooled and --val-mode cyclone are also available.

 4. v1 called zero_grad() once per epoch, accumulated over all ~10
    training graphs, and stepped once — 300 epochs = 300 Adam
    steps, and early stopping usually cut that to well under 100.
    Default is now one step per graph in shuffled order (~10x the
    steps for identical compute). --step-mode accumulate restores
    the old behaviour.

 5. Depth head was an unbounded Linear and emitted negative depths
    (~1.4% of predictions). Now wrapped in softplus.

 6. Depth MAE was oracle-conditioned (scored only where the road
    truly flooded). The deployed score is P(flood) x depth, so an
    end-to-end risk MAE over dry + flooded-with-depth nodes is now
    reported alongside it.

 7. The checkpoint did not carry mu/sd, so it could not be used for
    inference. Scaler, feature names and threshold now travel with
    the weights.

 8. Misc: float(loss) on a grad-tracking tensor (warning);
    --weight-decay and --val-cyclone were accepted by the functions
    but not exposed on the CLI; epochs_run NameError'd at
    --epochs 0; heads was logged for sage/gcn where it does nothing.

Usage
-----
    # one fold, one seed — start here
    python train_gat.py --test-city Puri --seed 0

    # everything: 4 folds x 5 seeds
    python train_gat.py --all

    # sanity-check the scaler on your real data before a long run
    python train_gat.py --check-features

Outputs results/<model>__test_<city>__seed<k>.json and the
checkpoint per run.
================================================================
"""
import os
import glob
import json
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv, GCNConv
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score)

GRAPH_DIR   = "./graphs"
RESULTS_DIR = "./results"
CKPT_DIR    = "./checkpoints"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# [FIX 1] Below this training std a column is treated as constant:
# we centre it but do not divide, instead of dividing by ~0.
STD_FLOOR = 1e-2
# [FIX 1] Hard clip on scaled features. Even with the floor, a
# held-out city can sit far outside the training range; +/-10 sigma
# is generous and stops one column from dominating the first layer.
SCALED_CLIP = 10.0

# Fallback if graphs/manifest.json has no feature names.
DEFAULT_FEATURES = [
    'hand_m', 'slope_deg', 'elev_rank', 'road_length_m', 'degree',
    'hand_min_1hop', 'hand_mean_1hop', 'elev_rank_mean_1hop',
    'rainfall_mm', 'effective_rain_mm', 'runoff_coeff', 'curve_number',
    'hw_motorway', 'hw_trunk', 'hw_primary', 'hw_secondary', 'hw_tertiary',
    'hw_residential', 'hw_unclassified', 'hw_service', 'hw_other']


# ================================================================
# 1. DATA
# ================================================================
def load_graphs(graph_dir=GRAPH_DIR):
    """
    Loads .pt if present, otherwise falls back to the .npz files
    build_graphs_v2.py writes when torch_geometric was unavailable
    at build time. Saves re-running the builder over ~1.9M points.

    weights_only=False is required for .pt: torch>=2.6 defaults to
    True and PyG Data objects are not in the default allowlist.
    These are files you generated yourself, so it is safe here.
    """
    pt = sorted(glob.glob(os.path.join(graph_dir, "*.pt")))
    if pt:
        graphs = [torch.load(p, weights_only=False) for p in pt]
    else:
        npz = sorted(glob.glob(os.path.join(graph_dir, "*.npz")))
        if not npz:
            raise FileNotFoundError(
                f"No .pt or .npz graphs in {graph_dir}. "
                f"Run build_graphs_v2.py first.")
        print(f"No .pt found — loading {len(npz)} .npz graphs instead.")
        graphs = []
        for p in npz:
            d = np.load(p, allow_pickle=True)
            g = Data(
                x=torch.tensor(d["x"], dtype=torch.float),
                edge_index=torch.tensor(d["edge_index"], dtype=torch.long),
                edge_attr=torch.tensor(d["edge_attr"], dtype=torch.float),
                y_flood=torch.tensor(d["y_flood"], dtype=torch.long),
                mask_flood=torch.tensor(d["mask_flood"], dtype=torch.bool),
                y_depth=torch.tensor(d["y_depth"], dtype=torch.float),
                mask_depth=torch.tensor(d["mask_depth"], dtype=torch.bool),
            )
            g.city = str(d["city"])
            g.cyclone = str(d["cyclone"])
            graphs.append(g)

    print(f"Loaded {len(graphs)} graphs: "
          + ", ".join(f"{g.city}/{g.cyclone}" for g in graphs))
    return graphs


def load_feature_names(graph_dir, n_feat):
    """Feature names make the scaler diagnostics readable. Optional."""
    mf = os.path.join(graph_dir, "manifest.json")
    if os.path.exists(mf):
        try:
            with open(mf) as f:
                man = json.load(f)
            for key in ("features", "feature_names", "node_features"):
                if key in man and len(man[key]) == n_feat:
                    return list(man[key])
        except Exception:
            pass
    if len(DEFAULT_FEATURES) == n_feat:
        return list(DEFAULT_FEATURES)
    return [f"f{i}" for i in range(n_feat)]


def make_fold(graphs, test_city, val_mode="fixed", val_cyclone=None,
              val_frac=0.2):
    """
    test  = every event of the held-out city
    val   = a slice of the TRAINING cities (never the test city)
    train = the rest

    Early stopping on the test city would be leakage — you would be
    choosing the epoch that happens to look best on the thing you
    are about to report. So validation comes from training cities.

    [FIX 3] READ THIS BEFORE QUOTING A VAL NUMBER. Whatever mode you
    pick, the val graphs share IDENTICAL node sets with training
    graphs — same city, same roads, different storm. So val is not
    just "optimistic because it is a seen city"; it is partly
    memorisation of those exact nodes. Measured gap on the Kolkata
    fold: val PR-AUC 0.73 vs test PR-AUC 0.36. Val is a training
    diagnostic here, not an estimate of generalisation.

    Modes:
      fixed    val is still built and logged, but the checkpoint is
               the final model after a fixed step budget. Nothing is
               selected on val. This is the default and the easiest
               to defend.
      pooled   deterministic ~val_frac slice of the training events,
               spread across cities (every k-th event in sorted
               order). Matches the agreed team protocol; keeps early
               stopping.
      cyclone  hold out one named cyclone from the training cities.
               Pass --val-cyclone explicitly; the v1 default of
               "last alphabetically" landed on Yaas in all four
               folds and on the 31.9% outlier for test=Kolkata.

    CAVEAT worth stating in the paper: val is "new storm, city (and
    roads) the model has seen", while test is "entirely new city".
    Different distributions. With only 4 cities, holding out a
    second city for validation would leave just 2 for training,
    which is worse — hence --val-mode fixed as the default.
    """
    test = [g for g in graphs if g.city == test_city]
    pool = [g for g in graphs if g.city != test_city]
    if not test:
        raise ValueError(f"No graphs for test city {test_city!r}. "
                         f"Available: {sorted({g.city for g in graphs})}")

    pool = sorted(pool, key=lambda g: (g.city, g.cyclone))

    if val_mode == "cyclone":
        if val_cyclone is None:
            # deterministic, but see the docstring — pin it yourself
            val_cyclone = sorted({g.cyclone for g in pool})[-1]
            print(f"  [warn] --val-mode cyclone with no --val-cyclone; "
                  f"defaulting to {val_cyclone!r}. Pin this explicitly.")
        val   = [g for g in pool if g.cyclone == val_cyclone]
        train = [g for g in pool if g.cyclone != val_cyclone]
        tag = f"cyclone={val_cyclone}"
    else:
        # 'pooled' and 'fixed' both use the deterministic slice.
        # Under 'fixed' it is only used for logging.
        k = max(1, int(round(1.0 / max(val_frac, 1e-9))))
        val   = [g for i, g in enumerate(pool) if i % k == k - 1]
        train = [g for i, g in enumerate(pool) if i % k != k - 1]
        if not val:                      # tiny pool guard
            val, train = pool[-1:], pool[:-1]
        tag = f"pooled ~{100 * len(val) / len(pool):.0f}% of events"

    if not train:
        raise ValueError("Validation slice consumed the whole training set.")

    print(f"\nFold  test={test_city} ({len(test)} events)  "
          f"val={len(val)} ({tag})  train={len(train)}  [mode={val_mode}]")
    print("      val   = " + ", ".join(f"{g.city}/{g.cyclone}" for g in val))
    print("      train = " + ", ".join(f"{g.city}/{g.cyclone}" for g in train))
    return train, val, test


def fit_scaler(train_graphs, feature_names=None, verbose=True):
    """
    Mean/std from TRAINING graphs only. Fitting on all graphs would
    leak the test city's feature distribution into every fold.

    [FIX 1] v1 did .clamp(min=1e-6). That is the single most
    damaging bug in the original script. A one-hot road class that
    happens to be absent from all three training cities has training
    std exactly 0; the held-out city then gets (1 - 0) / 1e-6 = 1e6
    fed into layer 1, and the fold is destroyed silently — no error,
    just a bad number. curve_number is the other candidate: the
    build log reports 1.3 unique values per graph.

    Fix: floor the std at STD_FLOOR and centre-only below it. The
    floored columns are printed, so if this fires on your real data
    you will see it rather than wonder why one fold underperforms.
    """
    X = torch.cat([g.x for g in train_graphs], dim=0)
    mu = X.mean(dim=0)
    sd_raw = X.std(dim=0)
    floored = sd_raw < STD_FLOOR
    sd = torch.where(floored, torch.ones_like(sd_raw), sd_raw)

    if verbose and bool(floored.any()):
        names = feature_names or [f"f{i}" for i in range(len(sd_raw))]
        cols = [f"{names[i]} (std={float(sd_raw[i]):.2e})"
                for i in torch.nonzero(floored).flatten().tolist()]
        print(f"  [scaler] {len(cols)} near-constant column(s) in TRAIN — "
              f"centred, not divided:")
        for c in cols:
            print(f"           - {c}")
        print("           if a hw_* column is here, that road class is "
              "absent from the training cities;")
        print("           v1 would have scaled it to ~1e6 in the test city.")
    return mu, sd


def apply_scaler(graphs, mu, sd, device):
    out = []
    for g in graphs:
        g = g.clone()
        # [FIX 1] clip: a held-out city can sit far outside the
        # training range even with a sane std.
        g.x = ((g.x - mu) / sd).clamp(-SCALED_CLIP, SCALED_CLIP)
        out.append(g.to(device))
    return out


def report_scaled_range(name, graphs):
    mx = max(float(g.x.abs().max()) for g in graphs)
    print(f"  scaled |x| max ({name}): {mx:.3g}")
    return mx


def pos_weight_from(train_graphs, verbose=True):
    """
    ~99% of roads are dry. Unweighted, the model predicts "dry"
    everywhere and looks excellent on accuracy while being useless.
    pos_weight = n_negative / n_positive rebalances the loss.

    NOTE: this differs per fold (measured 10.1-11.2 across the four
    folds), so the effective flood:depth loss balance shifts between
    folds even at a fixed --lambda-depth. Both loss components are
    logged so you can see it.
    """
    pos = neg = 0
    for g in train_graphs:
        y = g.y_flood[g.mask_flood]
        pos += int((y == 1).sum())
        neg += int((y == 0).sum())
    if pos == 0:
        raise ValueError("No positive flood labels in the training set.")
    w = neg / pos
    if verbose:
        print(f"  class balance: {pos:,} flooded / {neg:,} dry  "
              f"-> pos_weight={w:.1f}")
    return torch.tensor(w, dtype=torch.float)


# ================================================================
# 2. MODEL
# ================================================================
class FloodNet(nn.Module):
    """
    Two GNN layers, then two heads.

    Depth 2 is deliberate. A road line-graph has mean degree ~3-4,
    so 2 layers already reaches ~2 intersections out. Going deeper
    oversmooths quickly on low-degree graphs — every node's vector
    drifts toward the graph average and distinctions vanish. Depth
    is a genuine ablation axis (2 vs 3 vs 4), arguably more
    interesting here than which convolution you pick.
    """
    def __init__(self, in_dim, hidden=64, heads=4, edge_dim=3,
                 dropout=0.3, conv="gatv2"):
        super().__init__()
        self.conv_type = conv
        self.dropout = dropout
        if conv == "gatv2":
            # concat=True -> layer 1 outputs hidden*heads
            self.conv1 = GATv2Conv(in_dim, hidden, heads=heads,
                                   edge_dim=edge_dim, dropout=dropout)
            self.conv2 = GATv2Conv(hidden * heads, hidden, heads=1,
                                   edge_dim=edge_dim, dropout=dropout)
            out_dim = hidden
        elif conv == "sage":
            self.conv1 = SAGEConv(in_dim, hidden)
            self.conv2 = SAGEConv(hidden, hidden)
            out_dim = hidden
        elif conv == "gcn":
            self.conv1 = GCNConv(in_dim, hidden)
            self.conv2 = GCNConv(hidden, hidden)
            out_dim = hidden
        else:
            raise ValueError(f"unknown conv {conv!r}")
        self.flood_head = nn.Linear(out_dim, 1)   # logit
        self.depth_head = nn.Linear(out_dim, 1)   # pre-softplus

    def forward(self, x, edge_index, edge_attr=None):
        kw = {}
        if self.conv_type == "gatv2" and edge_attr is not None:
            kw["edge_attr"] = edge_attr
        h = self.conv1(x, edge_index, **kw)
        h = F.elu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, edge_index, **kw)
        h = F.elu(h)
        flood = self.flood_head(h).squeeze(-1)
        # [FIX 5] Water depth cannot be negative. v1's bare Linear
        # emitted negatives on ~1.4% of nodes, which then propagate
        # straight into P(flood) x depth as negative risk.
        depth = F.softplus(self.depth_head(h)).squeeze(-1)
        return flood, depth


# ================================================================
# 3. LOSS
# ================================================================
def compute_loss(flood_logit, depth_pred, g, pos_weight, lambda_depth=0.5):
    """
    Masked on both heads. mask=False means UNKNOWN, not negative —
    those nodes must not contribute to the loss at all.

    Flood head: BCE over every node with a SAR observation.
    Depth head: Huber (smooth L1) over nodes that flooded AND where
    FwDET actually assigned a water surface. Huber rather than MSE
    because a handful of deep outliers should not dominate.
    """
    mf = g.mask_flood
    loss_flood = F.binary_cross_entropy_with_logits(
        flood_logit[mf], g.y_flood[mf].float(), pos_weight=pos_weight)

    md = g.mask_depth
    if md.sum() > 0:
        loss_depth = F.smooth_l1_loss(depth_pred[md], g.y_depth[md])
    else:
        loss_depth = torch.zeros((), device=flood_logit.device)

    return loss_flood + lambda_depth * loss_depth, loss_flood, loss_depth


# ================================================================
# 4. EVALUATION
# ================================================================
@torch.no_grad()
def collect_predictions(model, graphs):
    """One pass, everything downstream reads from these arrays."""
    model.eval()
    P, Y, DP, DY, RP, RY = [], [], [], [], [], []
    for g in graphs:
        logit, depth = model(g.x, g.edge_index, getattr(g, "edge_attr", None))
        p = torch.sigmoid(logit)

        mf = g.mask_flood
        P.append(p[mf].cpu().numpy())
        Y.append(g.y_flood[mf].cpu().numpy())

        md = g.mask_depth
        if md.sum() > 0:
            DP.append(depth[md].cpu().numpy())
            DY.append(g.y_depth[md].cpu().numpy())

        # [FIX 6] End-to-end risk. The deployed quantity is
        # P(flood) x depth, so score that too — not just depth given
        # that it flooded, which is an oracle you will not have at
        # inference time. Nodes that flooded but where FwDET assigned
        # no water surface are UNKNOWN, so they are excluded rather
        # than counted as zero risk.
        known = mf & (~g.y_flood.bool() | md)
        if known.sum() > 0:
            RP.append((p[known] * depth[known]).cpu().numpy())
            RY.append((g.y_flood[known].float()
                       * g.y_depth[known]).cpu().numpy())

    out = {"p": np.concatenate(P), "y": np.concatenate(Y)}
    out["dp"] = np.concatenate(DP) if DP else None
    out["dy"] = np.concatenate(DY) if DY else None
    out["rp"] = np.concatenate(RP) if RP else None
    out["ry"] = np.concatenate(RY) if RY else None
    return out


def metrics_from(pred, threshold=0.5):
    """
    PR-AUC (average precision) is the headline number. ROC-AUC is
    optimistic on heavily imbalanced data; accuracy is meaningless
    here (predict "dry" everywhere -> ~99%).

    Recall on the flooded class is the one that matters
    operationally: a missed flooded road is the error that hurts
    someone.

    [FIX 2] precision/recall/f1 are reported at the threshold you
    pass in, and the caller passes one tuned on validation. At the
    naive 0.5 with pos_weight ~10 you measure the pos_weight, not
    the model — v1 read recall 0.74 / precision 0.28 for that reason.
    Threshold-free metrics (PR-AUC, ROC-AUC) are unaffected.
    """
    p, y = pred["p"], pred["y"]
    hard = (p >= threshold).astype(int)
    m = {"n": int(len(y)), "n_pos": int(y.sum()), "threshold": float(threshold),
         "n_flagged": int(hard.sum()), "flag_rate": float(hard.mean())}
    if hard.sum() == 0:
        # Guarded against in pick_operating_point, but if it ever
        # happens the metrics below are all 0 and mean nothing.
        m["degenerate_all_negative"] = True

    if y.sum() == 0 or y.sum() == len(y):
        # a fold can contain an event with no positives at all
        m.update(pr_auc=float("nan"), roc_auc=float("nan"),
                 precision=float("nan"), recall=float("nan"),
                 f1=float("nan"), pr_auc_baseline=float("nan"))
    else:
        m["pr_auc"]    = float(average_precision_score(y, p))
        m["roc_auc"]   = float(roc_auc_score(y, p))
        m["precision"] = float(precision_score(y, hard, zero_division=0))
        m["recall"]    = float(recall_score(y, hard, zero_division=0))
        m["f1"]        = float(f1_score(y, hard, zero_division=0))
        # baseline: what you would get by guessing at the base rate
        m["pr_auc_baseline"] = float(y.mean())

    if pred["dp"] is not None:
        dp, dy = pred["dp"], pred["dy"]
        m["depth_mae"]  = float(np.abs(dp - dy).mean())
        m["depth_rmse"] = float(np.sqrt(((dp - dy) ** 2).mean()))
        m["depth_n"]    = int(len(dy))
        m["depth_pred_min"] = float(dp.min())          # must be >= 0 now

    if pred["rp"] is not None:
        rp, ry = pred["rp"], pred["ry"]
        m["risk_mae"]  = float(np.abs(rp - ry).mean())
        m["risk_rmse"] = float(np.sqrt(((rp - ry) ** 2).mean()))
        m["risk_n"]    = int(len(ry))
    return m


@torch.no_grad()
def evaluate(model, graphs, threshold=0.5):
    return metrics_from(collect_predictions(model, graphs), threshold)


@torch.no_grad()
def pick_operating_point(model, graphs, objective="f1", min_precision=0.3,
                         verbose=True):
    """
    [FIX 2] Choose the operating point on VALIDATION, never on test.

    objective='f1'         maximise F1.
    objective='recall@p'   maximise recall subject to precision >=
                           min_precision. Closer to what an
                           evacuation router wants: missing a flooded
                           road is worse than flagging a dry one, but
                           not infinitely worse.

    Returns (threshold, positive_rate). The rate is what
    --threshold-mode rate uses downstream.

    Two guards, both of which fired in testing:

      a) A threshold in the extreme tail can look great on val and
         then predict NOTHING on the test city, giving a silent
         precision=0 / recall=0 with no error. Grid points that flag
         fewer than 20% of the val base rate are skipped.

      b) If no threshold reaches min_precision under 'recall@p', fall
         back to the F1 optimum and say so, rather than returning a
         degenerate corner of the grid.
    """
    pred = collect_predictions(model, graphs)
    p, y = pred["p"], pred["y"]
    if y.sum() == 0 or y.sum() == len(y):
        return 0.5, float((p >= 0.5).mean())

    base = float(y.mean())
    min_flag_rate = 0.2 * base                       # guard (a)
    grid = np.unique(np.quantile(p, np.linspace(0.05, 0.9995, 400)))

    best_f1, t_f1 = -np.inf, 0.5
    best_rec, t_rec = -np.inf, None
    for t in grid:
        hard = (p >= t).astype(int)
        if hard.mean() < min_flag_rate:
            continue
        prec = precision_score(y, hard, zero_division=0)
        rec  = recall_score(y, hard, zero_division=0)
        f1v  = 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)
        if f1v > best_f1:
            best_f1, t_f1 = f1v, float(t)
        if prec >= min_precision and rec > best_rec:
            best_rec, t_rec = rec, float(t)

    if objective == "recall@p":
        if t_rec is None:                            # guard (b)
            if verbose:
                print(f"  [warn] no val threshold reaches precision "
                      f">= {min_precision}; falling back to max-F1.")
            t = t_f1
        else:
            t = t_rec
    else:
        t = t_f1
    return t, float((p >= t).mean())


# ================================================================
# 5. TRAIN ONE FOLD
# ================================================================
def train_one(graphs, test_city, seed=0, conv="gatv2", hidden=64, heads=4,
              dropout=0.3, lr=5e-3, weight_decay=5e-4, epochs=300,
              patience=40, lambda_depth=0.5, device="cpu", verbose=True,
              val_mode="fixed", val_cyclone=None, val_frac=0.2,
              step_mode="per_graph", threshold_objective="f1",
              threshold_mode="rate",
              min_precision=0.3, feature_names=None, graph_dir=GRAPH_DIR):
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    train_g, val_g, test_g = make_fold(graphs, test_city, val_mode=val_mode,
                                       val_cyclone=val_cyclone,
                                       val_frac=val_frac)

    if feature_names is None:
        feature_names = load_feature_names(graph_dir, train_g[0].x.shape[1])

    mu, sd = fit_scaler(train_g, feature_names, verbose=verbose)
    train_g = apply_scaler(train_g, mu, sd, device)
    val_g   = apply_scaler(val_g,   mu, sd, device)
    test_g  = apply_scaler(test_g,  mu, sd, device)
    if verbose:
        report_scaled_range("train", train_g)
        report_scaled_range("test ", test_g)

    pw = pos_weight_from(train_g, verbose=verbose).to(device)

    in_dim   = train_g[0].x.shape[1]
    edge_dim = (train_g[0].edge_attr.shape[1]
                if getattr(train_g[0], "edge_attr", None) is not None else None)

    model = FloodNet(in_dim, hidden=hidden, heads=heads, edge_dim=edge_dim,
                     dropout=dropout, conv=conv).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val, best_state, bad = -np.inf, None, 0
    ckpt = os.path.join(CKPT_DIR,
                        f"{conv}__test_{test_city.lower()}__seed{seed}.pt")
    n_steps = 0
    epoch = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        # Graphs are NOT small — Kolkata is 143k nodes / 545k edges —
        # but they still fit full-batch on a GPU, so no DataLoader.
        #
        # [FIX 4] v1 zeroed once, accumulated over all ~10 graphs and
        # stepped once per epoch: 300 epochs = 300 Adam steps, and
        # early stopping typically cut that below 100. Same compute,
        # ~10x the steps, if you step per graph in shuffled order.
        order = rng.permutation(len(train_g)) if step_mode == "per_graph" \
            else np.arange(len(train_g))
        tot = tot_f = tot_d = 0.0

        if step_mode == "accumulate":
            opt.zero_grad()
        for i in order:
            g = train_g[i]
            if step_mode == "per_graph":
                opt.zero_grad()
            logit, depth = model(g.x, g.edge_index,
                                 getattr(g, "edge_attr", None))
            loss, lf, ld = compute_loss(logit, depth, g, pw, lambda_depth)
            loss.backward()
            if step_mode == "per_graph":
                opt.step()
                n_steps += 1
            # [FIX 8] .detach(): float() on a grad-tracking tensor warns
            tot   += float(loss.detach())
            tot_f += float(lf.detach())
            tot_d += float(ld.detach())
        if step_mode == "accumulate":
            opt.step()
            n_steps += 1

        val_m = evaluate(model, val_g)
        score = val_m.get("pr_auc", float("nan"))
        history.append({"epoch": epoch,
                        "train_loss": tot / len(train_g),
                        "val_pr_auc": score})

        if val_mode == "fixed":
            # [FIX 3] No selection on val. The reported model is the
            # final one after the full budget. val_pr_auc is logged
            # as a training diagnostic only.
            best_val = score
        else:
            if np.isnan(score):
                score = -np.inf
            if score > best_val:
                best_val, bad = score, 0
                best_state = {k: v.detach().clone()
                              for k, v in model.state_dict().items()}
            else:
                bad += 1

        if verbose and epoch % 20 == 0:
            print(f"  epoch {epoch:4d}  steps {n_steps:5d}  "
                  f"loss {tot/len(train_g):.4f} "
                  f"(flood {tot_f/len(train_g):.4f} "
                  f"depth {tot_d/len(train_g):.4f})  "
                  f"val_pr_auc {val_m.get('pr_auc', float('nan')):.4f}")

        if val_mode != "fixed" and bad >= patience:
            if verbose:
                print(f"  early stop at epoch {epoch} "
                      f"(no val improvement for {patience})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    # [FIX 2] Operating point comes from validation.
    thr_val, rate_val = pick_operating_point(
        model, val_g, objective=threshold_objective,
        min_precision=min_precision, verbose=verbose)

    # Per-event test metrics as well as pooled — a pooled average
    # hides that one event may carry the whole fold.
    test_pred = collect_predictions(model, test_g)

    if threshold_mode == "rate":
        # Probabilities are not calibrated across cities — a held-out
        # city's whole distribution can sit above or below the val
        # city's, so an absolute cut transfers badly (it produced a
        # silent all-negative prediction in testing). Carrying the
        # predicted-positive RATE instead and re-deriving the cut from
        # the test city's own scores is far more stable, and it maps
        # onto how this gets deployed anyway: "flag the top X% riskiest
        # roads", where X is set by how many roads you can actually
        # sign, close or route around.
        thr = float(np.quantile(test_pred["p"], 1.0 - rate_val))
    else:
        thr = thr_val

    test_m    = metrics_from(test_pred, thr)
    test_m05  = metrics_from(test_pred, 0.5)      # kept for comparison
    val_m     = evaluate(model, val_g, thr_val)
    per_event = {f"{g.city}/{g.cyclone}": evaluate(model, [g], thr)
                 for g in test_g}

    # [FIX 7] The checkpoint has to carry the scaler or it cannot be
    # used for inference — normalisation is part of the model.
    torch.save({
        "state_dict": model.state_dict(),
        "mu": mu.cpu(), "sd": sd.cpu(),
        "feature_names": feature_names,
        "threshold": thr_val, "threshold_mode": threshold_mode,
        "val_flag_rate": rate_val,
        "arch": {"conv": conv, "in_dim": in_dim, "hidden": hidden,
                 "heads": heads, "edge_dim": edge_dim, "dropout": dropout},
        "scaled_clip": SCALED_CLIP, "std_floor": STD_FLOOR,
    }, ckpt)

    cfg = {"hidden": hidden, "dropout": dropout, "lr": lr,
           "weight_decay": weight_decay, "lambda_depth": lambda_depth,
           "epochs_run": epoch, "opt_steps": n_steps,
           "val_mode": val_mode, "step_mode": step_mode,
           "threshold_objective": threshold_objective,
           "threshold_mode": threshold_mode}
    if conv == "gatv2":
        cfg["heads"] = heads          # [FIX 8] meaningless for sage/gcn

    result = {
        "model": conv, "test_city": test_city, "seed": seed,
        "config": cfg,
        "val_pr_auc": float(best_val) if np.isfinite(best_val) else float("nan"),
        "val": val_m,
        "threshold": thr, "threshold_val": thr_val,
        "val_flag_rate": rate_val,
        "test": test_m,
        "test_at_0.5": test_m05,
        "test_per_event": per_event,
        "checkpoint": ckpt,
        "history": history,
    }
    out = os.path.join(RESULTS_DIR,
                       f"{conv}__test_{test_city.lower()}__seed{seed}.json")
    with open(out, "w") as f:
        json.dump(result, f, indent=2)

    print(f"  steps={n_steps}  threshold={thr:.4f} "
          f"[{threshold_mode}]  (val thr={thr_val:.4f}, val flag-rate="
          f"{rate_val:.4f}, objective={threshold_objective})")
    print(f"  TEST  pr_auc {test_m.get('pr_auc', float('nan')):.4f}  "
          f"(base rate {test_m.get('pr_auc_baseline', float('nan')):.4f})  "
          f"prec {test_m.get('precision', float('nan')):.4f}  "
          f"recall {test_m.get('recall', float('nan')):.4f}  "
          f"f1 {test_m.get('f1', float('nan')):.4f}")
    print(f"        depth_mae {test_m.get('depth_mae', float('nan')):.4f} "
          f"(oracle-conditioned)   "
          f"risk_mae {test_m.get('risk_mae', float('nan')):.4f} (end-to-end)")
    print(f"        at naive 0.5: prec "
          f"{test_m05.get('precision', float('nan')):.4f} "
          f"recall {test_m05.get('recall', float('nan')):.4f}")
    print(f"  VAL   pr_auc {val_m.get('pr_auc', float('nan')):.4f}  "
          f"<- same roads as train; expect this to overstate test")
    return result


# ================================================================
# 6. FEATURE SANITY CHECK
# ================================================================
def check_features(graphs, graph_dir=GRAPH_DIR):
    """
    Run this before a long job. It answers two questions:
      1. Does the v1 scaler bug fire on YOUR data? i.e. is any
         feature constant across three cities but present in the
         fourth. Look for hw_* columns.
      2. Which features actually vary WITHIN a graph? The build log
         shows rainfall_mm at ~64 unique values and curve_number at
         ~1.3 across 91k nodes — those are event-level constants and
         give the model an event intercept, not per-node signal.
         Worth an explicit ablation in the paper.
    """
    names = load_feature_names(graph_dir, graphs[0].x.shape[1])
    cities = sorted({g.city for g in graphs})

    print("\n" + "=" * 78)
    print("SCALER SAFETY — training std per LOCO fold "
          f"(floor = {STD_FLOOR})")
    print("=" * 78)
    any_hit = False
    for test_city in cities:
        tr = [g for g in graphs if g.city != test_city]
        te = [g for g in graphs if g.city == test_city]
        sd = torch.cat([g.x for g in tr], dim=0).std(dim=0)
        mu = torch.cat([g.x for g in tr], dim=0).mean(dim=0)
        bad = torch.nonzero(sd < STD_FLOOR).flatten().tolist()
        if not bad:
            print(f"  test={test_city:10s}  ok — no near-constant columns")
            continue
        any_hit = True
        print(f"  test={test_city:10s}  {len(bad)} near-constant column(s):")
        Xte = torch.cat([g.x for g in te], dim=0)
        for i in bad:
            v1 = float(((Xte[:, i] - mu[i]) / max(float(sd[i]), 1e-6))
                       .abs().max())
            print(f"      {names[i]:22s} train_std={float(sd[i]):.3e}  "
                  f"-> v1 would scale test to |x|max={v1:.3g}")
    if any_hit:
        print("\n  Any |x|max above ~50 means v1 silently destroyed that "
              "fold. v2 centres those\n  columns instead of dividing, so "
              "they contribute 0 rather than 1e6.")
    else:
        print("\n  Clean — but keep the floor: it costs nothing and the "
              "failure is silent.")

    print("\n" + "=" * 78)
    print("WITHIN-GRAPH VARIATION — is a feature per-node signal, or just "
          "an event intercept?")
    print("=" * 78)
    print("  ratio = (mean within-graph std) / (pooled std across all "
          "graphs).")
    print("  ~1.0 means the feature separates roads inside one city-event, "
          "which is what a")
    print("  per-node model needs. Near 0 means it is essentially one number "
          "per event.\n")
    Xall = torch.cat([g.x for g in graphs], dim=0)
    pooled = Xall.std(dim=0)
    print(f"  {'feature':24s} {'within-graph std':>18s} {'pooled std':>12s} "
          f"{'ratio':>8s} {'uniq/graph':>11s}")
    intercepts = []
    for i, nm in enumerate(names):
        w = float(np.mean([float(g.x[:, i].std()) for g in graphs]))
        p = float(pooled[i])
        u = float(np.mean([int(torch.unique(g.x[:, i]).numel())
                           for g in graphs]))
        ratio = w / p if p > 1e-12 else float("nan")
        flag = ""
        if np.isfinite(ratio) and ratio < 0.10:
            flag = "  <-- event-level, not per-node"
            intercepts.append(nm)
        print(f"  {nm:24s} {w:18.6f} {p:12.6f} {ratio:8.3f} "
              f"{u:11.1f}{flag}")
    if intercepts:
        print(f"\n  {len(intercepts)} feature(s) are event-level: "
              + ", ".join(intercepts))
        print("  They cannot separate two roads in the same city during the "
              "same storm — they")
        print("  only shift the whole graph. With ~10 training events that "
              "is an easy intercept")
        print("  to memorise. Run the ablation (drop them, rerun) and report "
              "the delta yourself;")
        print("  better that than a reviewer noticing.")


# ================================================================
# 7. CLI
# ================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--graph-dir", default=GRAPH_DIR)
    ap.add_argument("--test-city", default=None)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--conv", default="gatv2", choices=["gatv2", "sage", "gcn"])
    ap.add_argument("--hidden", type=int, default=64)
    ap.add_argument("--heads", type=int, default=4)
    ap.add_argument("--dropout", type=float, default=0.3)
    ap.add_argument("--lr", type=float, default=5e-3)
    ap.add_argument("--weight-decay", type=float, default=5e-4)   # [FIX 8]
    ap.add_argument("--epochs", type=int, default=300)
    ap.add_argument("--patience", type=int, default=40,
                    help="ignored when --val-mode fixed")
    ap.add_argument("--lambda-depth", type=float, default=0.5)
    # [FIX 3]
    ap.add_argument("--val-mode", default="fixed",
                    choices=["fixed", "pooled", "cyclone"],
                    help="fixed = no early stopping (default); "
                         "pooled = ~20%% slice of training events; "
                         "cyclone = hold out one named cyclone")
    ap.add_argument("--val-cyclone", default=None,
                    help="only with --val-mode cyclone; pin it explicitly")
    ap.add_argument("--val-frac", type=float, default=0.2)
    # [FIX 4]
    ap.add_argument("--step-mode", default="per_graph",
                    choices=["per_graph", "accumulate"])
    # [FIX 2]
    ap.add_argument("--threshold-objective", default="f1",
                    choices=["f1", "recall@p"])
    ap.add_argument("--threshold-mode", default="rate",
                    choices=["rate", "absolute"],
                    help="rate = carry the val positive-RATE and re-derive "
                         "the cut from the test city's own scores "
                         "(default, transfers across cities); "
                         "absolute = carry the raw probability cut")
    ap.add_argument("--min-precision", type=float, default=0.3,
                    help="only with --threshold-objective recall@p")
    ap.add_argument("--seeds", type=int, default=5,
                    help="number of seeds under --all")
    ap.add_argument("--all", action="store_true", help="every city x N seeds")
    ap.add_argument("--check-features", action="store_true",
                    help="scaler + feature-variance audit, then exit")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")
    graphs = load_graphs(args.graph_dir)
    cities = sorted({g.city for g in graphs})

    if args.check_features:
        check_features(graphs, args.graph_dir)
        return

    kw = dict(conv=args.conv, hidden=args.hidden, heads=args.heads,
              dropout=args.dropout, lr=args.lr,
              weight_decay=args.weight_decay, epochs=args.epochs,
              patience=args.patience, lambda_depth=args.lambda_depth,
              device=device, val_mode=args.val_mode,
              val_cyclone=args.val_cyclone, val_frac=args.val_frac,
              step_mode=args.step_mode,
              threshold_objective=args.threshold_objective,
              threshold_mode=args.threshold_mode,
              min_precision=args.min_precision, graph_dir=args.graph_dir)

    if args.all:
        rows = []
        for city in cities:
            for seed in range(args.seeds):
                print(f"\n{'='*62}\n{args.conv}  test={city}  seed={seed}")
                rows.append(train_one(graphs, city, seed=seed,
                                      verbose=False, **kw))

        print(f"\n{'='*84}\nSUMMARY — {args.conv}, mean +/- std over "
              f"{args.seeds} seeds\n")
        print(f"{'test city':11s} {'PR-AUC':>15s} {'base':>7s} "
              f"{'recall':>15s} {'F1':>15s} {'risk MAE':>15s}")
        for city in cities:
            r = [x for x in rows if x["test_city"] == city]

            def ms(key, sub="test"):
                v = [x[sub].get(key, float("nan")) for x in r]
                v = [x for x in v if not np.isnan(x)]
                return (f"{np.mean(v):.3f}±{np.std(v):.3f}" if v else "n/a")

            base = np.nanmean([x["test"].get("pr_auc_baseline", np.nan)
                               for x in r])
            print(f"{city:11s} {ms('pr_auc'):>15s} {base:7.3f} "
                  f"{ms('recall'):>15s} {ms('f1'):>15s} "
                  f"{ms('risk_mae'):>15s}")

        print("\nRead PR-AUC against the base rate column, not against 1.0.")
        print("Base rate is what random guessing scores. A PR-AUC of 0.25")
        print("on a 0.05 base rate is a 5x lift and genuinely good.")
        print("\nrecall/F1 are at the val-tuned threshold, not 0.5. The 0.5")
        print("numbers are still in each JSON under 'test_at_0.5' — they")
        print("measure pos_weight more than they measure the model.")
        print("\nrisk MAE is end-to-end P(flood) x depth, which is what you")
        print("actually deploy. 'depth_mae' in the JSON is conditioned on")
        print("the road having truly flooded — an oracle you will not have.")
        print("\nReport per-fold, never only the mean across cities — one")
        print("city carrying the average is a different result from four")
        print("cities agreeing.")
    else:
        city = args.test_city or cities[0]
        train_one(graphs, city, seed=args.seed, **kw)


if __name__ == "__main__":
    main()